# H3 角色替换：VDN8 → FastH3 OmniRef（CPU 转换与校验）

这个笔记只在 **CPU 运行时** 上做检查，不加载大模型，也不生成视频：

1. 恢复 `MyDrive/minimaxH3_colab` 里的环境快照。只挂载已有模型，不重装环境，也不下载模型。
2. 用私有仓库的转换脚本，把 VDN8 加速版的角色替换工作流换成 FastH3 OmniRef 8 步，并列出所有改动。
3. 对照 Comfy 官方的 FastH3 OmniRef 模板检查采样链，并确认模型文件都在、Character Swap LoRA 能挂上 FastH3。
4. 用固定版本的 ComfyUI 加载全部节点，然后调用它自己的 `validate_prompt` 校验整张图（只校验，不执行）。
5. 可选：在 CPU 上打开 ComfyUI 画布，看看转换后的工作流长什么样。

从上往下依次运行即可。

In [ ]:
import shutil, subprocess, sys

gpu = ''
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
print('Python', sys.version.split()[0])
if gpu:
    raise RuntimeError(f'当前是 GPU 运行时：{gpu}\n本笔记只做转换和校验，请在「代码执行程序 → 更改运行时类型」里改成 CPU。')
print('CPU 运行时 ✓')

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/minimaxH3_colab')
if not (DRIVE_ROOT / 'runtime-cache').is_dir():
    raise RuntimeError(f'没找到 {DRIVE_ROOT}/runtime-cache，请确认挂载的是存放 H3 环境的那个 Google 账号。')
print('Drive root:', DRIVE_ROOT)

## 1. 恢复环境（不下载模型）

先确认 Drive 里有当前 Python 版本对应的环境快照，没有就停下，避免重装整套环境。启动脚本加了 `--skip-models`，只恢复快照、挂载模型，不下载，也不对 22GB 的大模型做 SHA-256 校验。

In [ ]:
import json, subprocess, sys
from pathlib import Path

def run(cmd, cwd=None):
    """Run a command and stream its output into this cell."""
    cmd = [str(part) for part in cmd]
    print('+', ' '.join(cmd))
    proc = subprocess.Popen(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'命令失败，退出码 {proc.returncode}')

LAUNCHER = Path('/content/minimaxH3_colab_launcher')
if not LAUNCHER.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/SydneyMCDonaldbigking/minimaxH3_colab.git', LAUNCHER])
BOOTSTRAP = LAUNCHER / 'colab' / 'bootstrap_colab.py'

plan = json.loads(subprocess.run([sys.executable, BOOTSTRAP, '--drive-root', DRIVE_ROOT, '--plan'],
                                 capture_output=True, text=True, check=True).stdout)
snapshot = Path(plan['runtime_snapshot'])
if not snapshot.is_file():
    raise RuntimeError(f'Drive 里没有当前 Python 对应的环境快照 {snapshot.name}；为免重装整套环境，先停在这里。')
print(f'环境快照 ✓ {snapshot.name}（{snapshot.stat().st_size / 1024 ** 3:.2f} GiB）')

run([sys.executable, BOOTSTRAP, '--drive-root', DRIVE_ROOT,
     '--profile', 'dance_swap', '--prepare-on-cpu', '--skip-models'])
COMFY = Path('/content/ComfyUI')
SOURCE = Path('/content/comfyui_quick_workflow')
PY = '/content/h3env/bin/python'

## 2. 原始 VDN8 工作流

原始工作流存放在 `MyDrive/minimaxH3_colab/assets/workflows/`。Drive 里没有的话，会先在会话存储（左侧「文件」面板）里找文件名带 `VDN8` 的 JSON，还找不到就弹出上传框。

In [ ]:
import hashlib, json

SRC_DIR = DRIVE_ROOT / 'assets' / 'workflows'
SRC = SRC_DIR / 'H3-Character-Swap-1.5-SAM3.1-VDN8.json'
if not SRC.is_file():
    candidates = sorted(Path('/content').glob('*VDN8*.json'))
    if candidates:
        name, data = candidates[0].name, candidates[0].read_bytes()
    else:
        from google.colab import files
        print('请上传原始工作流「H3+Character+Swap+1.5｜SAM+3.1+精准角色替换·VDN8+加速版.json」：')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError('请只上传一个 JSON 文件')
        name, data = next(iter(uploaded.items()))
    json.loads(data.decode('utf-8-sig'))
    SRC_DIR.mkdir(parents=True, exist_ok=True)
    SRC.write_bytes(data)
    print('已存到 Drive：', SRC, '（原文件名：', name, '）')
source_bytes = SRC.read_bytes()
print(f'{SRC}\n{len(source_bytes):,} bytes  sha256 {hashlib.sha256(source_bytes).hexdigest()}')

## 3. 转换：VDN8 → FastH3 OmniRef

转换用的是私有仓库 `comfyui_quick_workflow`（`codex/fast-h3-dance-swap` 分支）里的 `scripts/convert_vdn8_workflow_to_fasth3_omni.py`。它只替换 VDN 模型合成和执行计划这一段，SAM 3.1 跟踪、动态裁剪、遮罩 latent、局部回填和视频导出都原样保留。

转换结果存到 `MyDrive/minimaxH3_colab/workflows/`。如果那里已有内容不同的同名文件，会另存一份带时间戳的，不覆盖原文件。

In [ ]:
import shutil, time
import pandas as pd
from IPython.display import display

pd.set_option('display.max_colwidth', None)
CHECK = Path('/content/h3check'); CHECK.mkdir(exist_ok=True)
CONVERTER = SOURCE / 'scripts' / 'convert_vdn8_workflow_to_fasth3_omni.py'
CONVERTED = CHECK / 'H3-Character-Swap-1.5-SAM3.1-FastH3-OmniRef.json'
run([PY, CONVERTER, SRC, CONVERTED])
before = json.loads(source_bytes.decode('utf-8-sig'))
after = json.loads(CONVERTED.read_text(encoding='utf-8'))

repo_copy = SOURCE / 'workflows' / CONVERTED.name
print('与私有仓库里提交的成品一致：',
      repo_copy.is_file() and json.loads(repo_copy.read_text(encoding='utf-8-sig')) == after)

OUT_DIR = DRIVE_ROOT / 'workflows'; OUT_DIR.mkdir(exist_ok=True)
OUT = OUT_DIR / CONVERTED.name
if OUT.is_file() and OUT.read_bytes() != CONVERTED.read_bytes():
    OUT = OUT_DIR / f'{CONVERTED.stem}-{time.strftime("%Y%m%d-%H%M%S")}.json'
if not OUT.is_file():
    shutil.copy2(CONVERTED, OUT)
print('转换结果在 Drive：', OUT)
ui_workflows = COMFY / 'user' / 'default' / 'workflows'; ui_workflows.mkdir(parents=True, exist_ok=True)
shutil.copy2(CONVERTED, ui_workflows / CONVERTED.name)

def short(value, limit=90):
    text = json.dumps(value, ensure_ascii=False)
    return text if len(text) <= limit else text[:limit] + '…'

def widget_change(old, new):
    if isinstance(old, dict) and isinstance(new, dict):
        return '；'.join('删除嵌入的云端预览' if key == 'videopreview' else f'{key}: {short(old.get(key), 40)} → {short(new.get(key), 40)}'
                        for key in sorted(set(old) | set(new)) if old.get(key) != new.get(key))
    if isinstance(old, list) and isinstance(new, list) and len(old) == len(new):
        return '；'.join(f'[{i}] {short(a, 60)} → {short(b, 60)}' for i, (a, b) in enumerate(zip(old, new)) if a != b)
    return f'{short(old)} → {short(new)}'

old_nodes = {node['id']: node for node in before['nodes']}
new_nodes = {node['id']: node for node in after['nodes']}
rows = []
for i in sorted(set(old_nodes) - set(new_nodes)):
    node = old_nodes[i]
    rows.append(['删除', i, node['type'], node.get('title') or '', short(node.get('widgets_values'))])
for i in sorted(set(new_nodes) - set(old_nodes)):
    node = new_nodes[i]
    rows.append(['新增', i, node['type'], node.get('title') or '', short(node.get('widgets_values'), 120)])
for i in sorted(set(old_nodes) & set(new_nodes)):
    old, new = old_nodes[i], new_nodes[i]
    if old.get('widgets_values') != new.get('widgets_values'):
        change = '说明文字更新' if new['type'] == 'MarkdownNote' else widget_change(old.get('widgets_values'), new.get('widgets_values'))
        rows.append(['改参数', i, new['type'], new.get('title') or '', change])
print(f'\n节点 {len(before["nodes"])} → {len(after["nodes"])}，连线 {len(before["links"])} → {len(after["links"])}')
display(pd.DataFrame(rows, columns=['变动', '节点', '类型', '标题', '内容']))

def sampling_wiring(workflow):
    by_id = {node['id']: node for node in workflow['nodes']}
    links = {link[0]: link for link in workflow['links']}
    def describe(node_id):
        node = by_id[node_id]
        return f"{node['type']} #{node_id} {short(node.get('widgets_values') or [], 70)}"
    def model_path(node_id):
        path = []
        while True:
            node = by_id[node_id]
            path.append(node['type'])
            feeds = [links[i['link']][1] for i in node.get('inputs', []) if i.get('type') == 'MODEL' and i.get('link') in links]
            if not feeds:
                return ' → '.join(reversed(path))
            node_id = feeds[0]
    sampler = next(node for node in workflow['nodes'] if node['type'] == 'SamplerCustomAdvanced')
    rows = []
    for item in sampler['inputs']:
        source = links[item['link']][1]
        rows.append([item['name'], describe(source)])
        if item['name'] in ('guider', 'sigmas'):
            rows.append([f'{item["name"]} 的模型链', model_path(source)])
    return rows

print('\n采样部分的接线（转换前 → 转换后）：')
wiring = [[name, a, b] for (name, a), (_, b) in zip(sampling_wiring(before), sampling_wiring(after))]
display(pd.DataFrame(wiring, columns=['采样器输入', '转换前（VDN8）', '转换后（FastH3 OmniRef）']))

## 4. 静态检查

这一步不启动 ComfyUI，检查五件事：连线是否完整、VDN 节点是否已删干净、采样链是否和 Comfy 官方 FastH3 OmniRef 模板一致（模板固定到 `Comfy-Org/workflow_templates@1524172`）、工作流用到的模型文件是否都在、Character Swap LoRA 的每个目标层在 FastH3 里是否存在且形状一致（只读文件头）。

In [ ]:
import re, struct, urllib.request

checks = []
def check(item, ok, detail=''):
    checks.append(['✅' if ok else '❌', item, detail])

by_id = {node['id']: node for node in after['nodes']}
def types_match(a, b):
    return '*' in (a, b) or bool(set(str(a).split(',')) & set(str(b).split(',')))
bad = []
for link_id, src, src_slot, dst, dst_slot, kind in after['links']:
    if src not in by_id or dst not in by_id:
        bad.append(f'{link_id}: 端点不存在'); continue
    out, inp = by_id[src]['outputs'][src_slot], by_id[dst]['inputs'][dst_slot]
    if link_id not in (out.get('links') or []) or inp.get('link') != link_id:
        bad.append(f'{link_id}: 两端记录不一致')
    if not (types_match(out['type'], kind) and types_match(inp['type'], kind)):
        bad.append(f'{link_id}: {out["type"]} → {kind} → {inp["type"]}')
check(f'连线完整、类型匹配（{len(after["links"])} 条）', not bad, '；'.join(bad[:5]))

left = [node['type'] for node in after['nodes'] if 'VDN' in node['type']]
check('VDN 节点已全部移除', not left, ', '.join(left))

OFFICIAL_URL = ('https://raw.githubusercontent.com/Comfy-Org/workflow_templates/'
                '1524172f454aff9c87b7e5ebc143af95542bf8f0/templates/video_fasth3_ominiref_r2v.json')
OFFICIAL_SHA256 = '38ac8a5b99035f85f240c3aec91ec9b898696097982ee3ba7cdc30f3bace1413'
raw = urllib.request.urlopen(OFFICIAL_URL, timeout=60).read()
official = json.loads(raw)
check('官方模板文件校验', hashlib.sha256(raw).hexdigest() == OFFICIAL_SHA256, 'video_fasth3_ominiref_r2v.json')

def widgets(workflow, kind):
    return [node.get('widgets_values') or [] for node in workflow['nodes'] if node['type'] == kind]
for kind, what in [('UNETLoader', '扩散模型'), ('MiniMaxH3SigmaShift', '视频/音频偏移'),
                   ('ModelAttentionBackend', '注意力后端'), ('BlockSparseAttention', 'VSA 稀疏注意力'),
                   ('KSamplerSelect', '采样器'), ('BasicScheduler', '调度器和步数'), ('BasicGuider', '引导方式（CFG 1）')]:
    mine, theirs = widgets(after, kind), widgets(official, kind)
    check(f'{what}与官方一致（{kind}）', mine == theirs, short(mine, 120))

def model_chain(workflow, end_type, skip=('LoraLoaderModelOnly', 'ModelPreviewOverrideKJ')):
    nodes_by_id = {node['id']: node for node in workflow['nodes']}
    links = {link[0]: link for link in workflow['links']}
    node = next(node for node in workflow['nodes'] if node['type'] == end_type)
    path = []
    while node is not None:
        path.append(node['type'])
        feeds = [links[i['link']][1] for i in node.get('inputs', []) if i.get('type') == 'MODEL' and i.get('link') in links]
        node = nodes_by_id[feeds[0]] if feeds else None
    return [kind for kind in reversed(path) if kind not in skip]
for end in ('BasicGuider', 'BasicScheduler'):
    mine, theirs = model_chain(after, end), model_chain(official, end)
    check(f'到 {end} 的模型链顺序与官方一致', mine == theirs,
          ' → '.join(mine) + '（另有 Character Swap LoRA 与 KJ 预览节点）')

LOADERS = {'UNETLoader': ('diffusion_models', 0), 'LoraLoaderModelOnly': ('loras', 0),
           'CheckpointLoaderSimple': ('checkpoints', 0), 'CLIPLoader': ('text_encoders', 0),
           'VAELoader': ('vae', 0), 'ModelPreviewOverrideKJ': ('vae_approx', 5)}
for node in after['nodes']:
    if node['type'] not in LOADERS or node.get('mode') == 4:
        continue
    folder, index = LOADERS[node['type']]
    name = node['widgets_values'][index]
    if name in ('none', ''):
        continue
    path = COMFY / 'models' / folder / name
    check(f'模型文件 {folder}/{name}', path.is_file(),
          f'{path.stat().st_size:,} bytes' if path.is_file() else '缺失')

def header(path):
    with open(path, 'rb') as handle:
        size = struct.unpack('<Q', handle.read(8))[0]
        return json.loads(handle.read(size))
lora_name = widgets(after, 'LoraLoaderModelOnly')[0][0]
unet_name = widgets(after, 'UNETLoader')[0][0]
lora = header(COMFY / 'models' / 'loras' / lora_name)
unet = header(COMFY / 'models' / 'diffusion_models' / unet_name)
targets = sorted({re.sub(r'\.(lora_[AB]\.weight|alpha)$', '', key).removeprefix('diffusion_model.')
                  for key in lora if key != '__metadata__'})
fits = [t for t in targets if t + '.weight' in unet
        and [lora[f'diffusion_model.{t}.lora_B.weight']['shape'][0], lora[f'diffusion_model.{t}.lora_A.weight']['shape'][1]]
            == unet[t + '.weight']['shape'][:2]]
check('Character Swap LoRA 能挂到 FastH3 上', len(fits) == len(targets),
      f'{len(fits)}/{len(targets)} 个目标层存在且形状一致（只读文件头）')

display(pd.DataFrame(checks, columns=['结果', '检查项', '说明']))

## 5. 补预览解码器 taeh3（只在缺失时）

原工作流的 `ModelPreviewOverrideKJ` 用 `taeh3.safetensors` 显示采样时的实时预览，这个文件要放在 `models/vae_approx`。缺了它，ComfyUI 会报 `taeh3.safetensors not in ['none']`，**主视频输出会被拒绝**。

如果 Drive 里还没有，这一格会从 madebyollin/taehv（MIT 许可，固定到提交 `62f7591`）下载 22.7MB 的 taeh3，存到 `MyDrive/minimaxH3_colab/models/vae_approx/`，并校验 SHA-256。启动脚本会把这个目录挂进 ComfyUI；旧版启动脚本不挂，这一格会补一个链接。

In [ ]:
TAEH3_URL = ('https://raw.githubusercontent.com/madebyollin/taehv/'
             '62f7591f59dfbb4c3c02b7a621d180a9eeaba26c/safetensors/taeh3.safetensors')
TAEH3_SHA256 = '4fd022bfcab08772fe0536b17ea1a3bbb5625be11e397868d1c5d891863d4c13'

wanted = {node['widgets_values'][5] for node in after['nodes'] if node['type'] == 'ModelPreviewOverrideKJ'} - {'none', ''}
local = COMFY / 'models' / 'vae_approx' / 'taeh3.safetensors'
stored = DRIVE_ROOT / 'models' / 'vae_approx' / 'taeh3.safetensors'
if not wanted:
    print('工作流没有用 tiny VAE，跳过。')
elif wanted != {'taeh3.safetensors'}:
    raise RuntimeError(f'工作流要的预览解码器不是 taeh3：{wanted}')
else:
    if not stored.is_file():
        stored.parent.mkdir(parents=True, exist_ok=True)
        partial = stored.with_name(stored.name + '.partial')
        urllib.request.urlretrieve(TAEH3_URL, partial)
        digest = hashlib.sha256(partial.read_bytes()).hexdigest()
        if digest != TAEH3_SHA256:
            partial.unlink()
            raise RuntimeError(f'taeh3 SHA-256 不符：{digest}')
        partial.replace(stored)
        print('已下载到 Drive：', stored)
    # New launchers link the whole vae_approx folder to Drive; older ones need a per-file link.
    if not local.exists():
        if local.is_symlink():
            local.unlink()
        local.parent.mkdir(parents=True, exist_ok=True)
        local.symlink_to(stored)
    print('已就位：', local, f'（{local.stat().st_size:,} bytes）')

## 6. ComfyUI 官方校验（CPU，只校验不执行）

单独起一个 Python 进程，用和 `main.py` 一样的方式加载 ComfyUI 和全部自定义节点，但不启动服务器，也不进队列。它做三件事：

1. 用节点的实际定义把画布转成 API 格式；
2. 逐个核对每个节点保存的参数，加载到当前版本时会不会错位；
3. 调用 ComfyUI 自己的 `execution.validate_prompt`。

校验需要原视频和人物图存在，所以会在**校验专用的** input 目录里放灰色占位图和 2 秒测试视频。ComfyUI 真正的 input 目录不受影响。

In [ ]:
%%writefile /content/h3check/validate_workflow.py
"""Check a ComfyUI canvas workflow against this ComfyUI install without running it.

Run from the ComfyUI root:  python validate_workflow.py WORKFLOW.json REPORT.json INPUT_DIR

Loads ComfyUI and every custom node the way main.py does, but never starts the
server or the queue.  The canvas is turned into an API prompt from the live node
definitions, each node's saved widgets are checked against the current widget
order, and the prompt goes through ComfyUI's own execution.validate_prompt.
"""
import asyncio
import json
import os
import sys
from pathlib import Path

workflow_path, report_path, input_dir = (Path(p).resolve() for p in sys.argv[1:4])
sys.path.insert(0, os.getcwd())
sys.argv = ["main.py", "--cpu", "--disable-assets", "--input-directory", str(input_dir)]

import main  # noqa: E402,F401  ComfyUI's module-level setup: paths, logging, prestartup scripts
import execution  # noqa: E402
import nodes  # noqa: E402
import server  # noqa: E402
from app.assets.manager import default_asset_manager  # noqa: E402
from comfy_api.internal import _ComfyNodeInternal  # noqa: E402

UI_ONLY = {"MarkdownNote", "Note", "PrimitiveNode", "Reroute"}
WIDGET_TYPES = {"INT", "FLOAT", "STRING", "BOOLEAN", "COMBO", "COMFY_DYNAMICCOMBO_V3"}


def node_info(class_type):
    cls = nodes.NODE_CLASS_MAPPINGS[class_type]
    if issubclass(cls, _ComfyNodeInternal):
        return cls.GET_NODE_INFO_V1()
    types = cls.INPUT_TYPES()
    return {"input": types, "input_order": {key: list(value) for key, value in types.items()}}


def spec_of(info, name):
    for group in ("required", "optional"):
        spec = info["input"].get(group, {}).get(name)
        if spec is not None:
            return spec
    return None


def is_widget(spec):
    return spec is not None and (isinstance(spec[0], list) or spec[0] in WIDGET_TYPES)


def has_control(spec):
    return bool(spec and len(spec) > 1 and isinstance(spec[1], dict) and spec[1].get("control_after_generate"))


def assign_like_frontend(info, values):
    """Hand widgets_values to widgets in definition order, as the frontend does on load."""
    assigned, stream = {}, iter(values)

    def take(name, spec):
        try:
            assigned[name] = next(stream)
        except StopIteration:
            return False
        if has_control(spec):
            next(stream, None)
        return True

    order = info["input_order"]
    for name in order.get("required", []) + order.get("optional", []):
        spec = spec_of(info, name)
        if not is_widget(spec):
            continue
        if not take(name, spec):
            break
        if spec[0] == "COMFY_DYNAMICCOMBO_V3":
            for option in spec[1].get("options", []):
                if option["key"] == assigned[name]:
                    for group in ("required", "optional"):
                        for sub, sub_spec in option["inputs"].get(group, {}).items():
                            take(f"{name}.{sub}", sub_spec)
    return assigned, list(stream)


def to_api(workflow):
    by_id = {node["id"]: node for node in workflow["nodes"]}
    links = {link[0]: link for link in workflow["links"]}
    prompt, widget_checks = {}, []

    def upstream(link_id):
        link = links.get(link_id)
        source = by_id.get(link[1]) if link else None
        if source is None or source.get("mode") == 2 or source["type"] in UI_ONLY - {"PrimitiveNode"}:
            return None
        if source["type"] == "PrimitiveNode":
            return ("value", source["widgets_values"][0])
        if source.get("mode") == 4:  # bypassed: pass the first same-typed input through
            for item in source.get("inputs", []):
                if item.get("link") is not None and item.get("type") == link[5]:
                    return upstream(item["link"])
            return None
        return ("link", [str(source["id"]), link[2]])

    for node in workflow["nodes"]:
        kind = node["type"]
        if kind in UI_ONLY or node.get("mode") in (2, 4) or kind not in nodes.NODE_CLASS_MAPPINGS:
            continue
        info = node_info(kind)
        values = node.get("widgets_values")
        if isinstance(values, dict):
            inputs = {key: value for key, value in values.items() if key != "videopreview"}
            widget_checks.append({"id": node["id"], "type": kind, "ok": True, "how": "按名称保存"})
        else:
            inputs, leftover = assign_like_frontend(info, values or [])
            saved = [item["widget"]["name"] for item in node.get("inputs", []) if item.get("widget")]
            saved = [name for name in saved if spec_of(info, name.split(".")[0]) is not None]
            named = node.get("widgets_values_named")
            if named is not None:
                ok, how = named == inputs, "对照 widgets_values_named"
            elif saved:
                ok, how = saved == list(inputs), "对照画布保存的控件名"
            else:
                ok, how = True, "无控件"
            widget_checks.append({
                "id": node["id"], "type": kind, "ok": ok, "how": how,
                "assigned": inputs, "expected": named if named is not None else saved,
                "leftover": leftover,
            })
        for item in node.get("inputs", []):
            if item.get("link") is None:
                continue
            source = upstream(item["link"])
            if source is None:
                inputs.pop(item["name"], None)
            else:
                inputs[item["name"]] = source[1]
        prompt[str(node["id"])] = {
            "class_type": kind,
            "inputs": inputs,
            "_meta": {"title": node.get("title") or kind},
        }
    return prompt, widget_checks


def main_check():
    workflow = json.loads(workflow_path.read_text(encoding="utf-8"))
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    server.PromptServer(loop, default_asset_manager())
    loop.run_until_complete(nodes.init_extra_nodes(init_custom_nodes=True, init_api_nodes=False))

    used = sorted({node["type"] for node in workflow["nodes"] if node["type"] not in UI_ONLY})
    missing = [kind for kind in used if kind not in nodes.NODE_CLASS_MAPPINGS]
    prompt, widget_checks = to_api(workflow)
    # validate_prompt reports ok as soon as one output survives, so list every output node too.
    output_nodes = sorted(
        key for key, item in prompt.items()
        if getattr(nodes.NODE_CLASS_MAPPINGS[item["class_type"]], "OUTPUT_NODE", False) is True
    )
    ok, error, outputs, node_errors = loop.run_until_complete(
        execution.validate_prompt("h3check", prompt, None)
    )
    all_ok = (not missing and all(c["ok"] for c in widget_checks) and ok
              and sorted(outputs) == output_nodes and not node_errors)
    report = {
        "all_ok": all_ok,
        "node_types_used": used,
        "missing_node_types": missing,
        "widget_checks": widget_checks,
        "output_nodes": output_nodes,
        "prompt": prompt,
        "validate_prompt": {"ok": ok, "error": error, "outputs": sorted(outputs), "node_errors": node_errors},
    }
    report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
    print(json.dumps({
        "all_ok": all_ok,
        "missing_node_types": missing,
        "widget_order_problems": [c["id"] for c in widget_checks if not c["ok"]],
        "outputs_passed": f"{len(outputs)}/{len(output_nodes)}",
        "nodes_with_errors": sorted(node_errors),
    }, ensure_ascii=False, indent=2, default=str))


# os._exit: custom nodes may leave non-daemon threads behind, and no server is meant to stay up.
status = 0
try:
    main_check()
except BaseException:
    import traceback
    traceback.print_exc()
    status = 1
sys.stdout.flush()
sys.stderr.flush()
os._exit(status)


In [ ]:
from PIL import Image

INPUT = CHECK / 'input'; INPUT.mkdir(exist_ok=True)
for node in after['nodes']:
    if node['type'] == 'LoadImage':
        Image.new('RGB', (512, 768), (128, 128, 128)).save(INPUT / node['widgets_values'][0])
    if node['type'] == 'VHS_LoadVideo':
        run(['ffmpeg', '-y', '-loglevel', 'error', '-f', 'lavfi', '-i', 'testsrc=size=512x288:rate=24',
             '-f', 'lavfi', '-i', 'sine=frequency=440:sample_rate=48000', '-t', '2', '-shortest',
             '-pix_fmt', 'yuv420p', INPUT / node['widgets_values']['video']])
REPORT = CHECK / 'validation_report.json'
run([PY, CHECK / 'validate_workflow.py', CONVERTED, REPORT, INPUT], cwd=COMFY)
report = json.loads(REPORT.read_text(encoding='utf-8'))
vp = report['validate_prompt']

summary = [
    ['✅' if not report['missing_node_types'] else '❌', '节点类型都已注册',
     f"{len(report['node_types_used'])} 种" + (f"，缺 {report['missing_node_types']}" if report['missing_node_types'] else '')],
    ['✅' if all(c['ok'] for c in report['widget_checks']) else '❌', '每个节点的参数加载后不错位',
     f"{sum(c['ok'] for c in report['widget_checks'])}/{len(report['widget_checks'])} 个节点"],
    ['✅' if vp['outputs'] == report['output_nodes'] else '❌', '所有输出节点通过 validate_prompt',
     f"{len(vp['outputs'])}/{len(report['output_nodes'])}：{vp['outputs']}"],
    ['✅' if not vp['node_errors'] else '❌', '没有节点报错', f"{len(vp['node_errors'])} 个"],
]
display(pd.DataFrame(summary, columns=['结果', '检查项', '说明']))
if vp['node_errors']:
    display(pd.DataFrame([[node_id, item['class_type'], e['message'], e['details']]
                          for node_id, item in vp['node_errors'].items() for e in item['errors']],
                         columns=['节点', '类型', '错误', '详情']))
new_ids = set(new_nodes) - set(old_nodes)
display(pd.DataFrame([[c['id'], c['type'], short(c.get('assigned'), 160)] for c in report['widget_checks'] if c['id'] in new_ids],
                     columns=['新增节点', '类型', '按当前节点定义加载后的参数']))
print('✅ 转换后的工作流通过全部校验' if report['all_ok'] else '❌ 还有问题，见上表')

## 7. 可选：在 CPU 上打开 ComfyUI 看画布

启动 ComfyUI（CPU 模式）之后，在左侧「工作流」里打开 `H3-Character-Swap-1.5-SAM3.1-FastH3-OmniRef.json` 就能看到转换后的画布。**只看，不要点运行**：CPU 运行时内存不够加载 FastH3。看完运行下一格关掉。

In [ ]:
import time, urllib.request
from google.colab import output

comfy_log = open(CHECK / 'comfyui.log', 'w')
comfy_server = subprocess.Popen([PY, 'main.py', '--cpu', '--disable-assets', '--listen', '127.0.0.1', '--port', '8188'],
                                cwd=COMFY, stdout=comfy_log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        urllib.request.urlopen('http://127.0.0.1:8188/system_stats', timeout=2)
        break
    except OSError:
        if comfy_server.poll() is not None:
            raise RuntimeError('ComfyUI 退出了，见 /content/h3check/comfyui.log')
        time.sleep(1)
output.serve_kernel_port_as_iframe(8188, height=900)

In [ ]:
comfy_server.terminate()
comfy_server.wait(timeout=60)
print('ComfyUI 已关闭')

## CPU 校验覆盖不到的地方（上 GPU 前看一眼）

- **生成效果、显存和速度**：要在 GPU 上实跑才能知道。尤其是 Character Swap LoRA 是在 H3 Ref2VA 基座上训练的，挂到 8 步蒸馏的 FastH3 上效果能不能保持，只能看成片。
- **文本编码器**：Drive 里只有 `qwen3vl_32b_minimax_h3_nvfp4_awq`。FastH3 官方说明里写，NVFP4 只有 Blackwell 显卡能加速（Colab 的 G4 就是 Blackwell）。用 A100、L4 或 H100 时，官方建议换成 INT8 或 BF16 编码器。
- **注意力后端**：`comfy kitchen attention` 只在 NVIDIA 或 AMD 显卡上出现。CPU 上的下拉框里没有这个选项，但节点自带校验函数会跳过检查，所以这里能通过。